<p align="center">
  <img src="foresight.png" alt="FORESIGHT" width="320">
</p>

# Deterministic forecast evaluation

This notebook shows how to use `ForecastPerformance` to evaluate **deterministic forecasts** with a suite of skill metrics.  



In [ ]:
import pandas as pd
from pathlib import Path
from performance import ForecastPerformance, Results 

test_dataset_path = Path(r'tests\test_datasets_hourly')

obs = pd.read_csv(test_dataset_path / 'observed.csv', sep=';', index_col=0, parse_dates=True)
obs.columns = pd.Index(pd.to_timedelta(obs.columns), name='leadtime')
obs.index.name = 'event_datetime'

det_path = test_dataset_path / 'det.parquet'
det = pd.read_parquet(det_path)
display(det.head(5))

In [ ]:
results = Results('Model', 'Metric', 'Leadtime')

fp = ForecastPerformance(obs)
fp.add(det, name='det 1')
fp.add(det + 2, name='det 2')
fp.add(det * 1.2, name='det 3')

for model in fp.names():
    print(f'{model}')
    for metric in [fp.KGEprime, fp.NSE, fp.RMSE, fp.MAE, fp.bias, fp.relative_bias, fp.Pearson]:
        for leadtime in det.index.get_level_values('leadtime').unique():
            results.append(Model=model, Metric=metric.__name__, Leadtime=leadtime,
                        Value=fp.deterministic(metric, model, leadtime=leadtime),
                        )
        
df = results.to_pandas(index=['Metric', 'Model'], columns=['Leadtime'])
display(df.iloc[:, ::6])